# Embedding checks from saved files
Reload the matrix and index independently. Check all mappings, permitted response joins, fresh last-token embeddings and a small semantic retrieval diagnostic. Held-out human answers stay unread.


In [1]:
import hashlib
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sentence_transformers import SentenceTransformer

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir())
def read_json(name):
    return json.loads((ROOT / name).read_text())
def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

config = read_json("configs/embedding_model.json")
audit = read_json("outputs/question_answer_embedding_audit.json")
bank = read_json("data/processed/question_bank.json")
items = {i["item_id"]: i for i in bank["items"]}
records = read_json("outputs/question_answer_index.json")
index = pd.DataFrame(records)
vectors = np.load(ROOT / "outputs/question_answer_embeddings.npy", allow_pickle=False)
checks = []
def check(name, condition):
    assert bool(condition), name
    checks.append(name)

for name, entry in audit["files"].items():
    check("saved file checksum: " + name, sha256(ROOT / name) == entry["sha256"])
for name, digest in audit["input_sha256"].items():
    check("input checksum: " + name, sha256(ROOT / name) == digest)
check("shape and dtype", vectors.shape == (1877, 1024) and vectors.dtype == np.float32)
check("finite unit vectors", np.isfinite(vectors).all() and np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-5))
check("unique vectors across all candidates", len(np.unique(vectors, axis=0)) == len(vectors))
check("contiguous unique row numbers", index["embedding_row"].tolist() == list(range(len(vectors))))
check("unique question/option keys", not index.duplicated(["item_id", "answer_index"]).any())
expected = [(i, o) for i in bank["items"] for o in i["options"]]
check("complete option coverage", len(records) == len(expected))
for record, (item, option) in zip(records, expected):
    assert (record["item_id"], record["answer_index"], record["response_code"]) == (item["item_id"], option["index"], option["code"])
    assert record["candidate_answer"] == option["label"]
    assert record["question_role"] == item["role"] and record["is_anchor"] == item["is_anchor"]
    expected_text = config["text_template"].format(instruction=config["instruction"], question_text=item["question_text"], code=option["code"], label=option["label"])
    assert record["input_text"] == expected_text
    assert record["text_sha256"] == hashlib.sha256(expected_text.encode()).hexdigest()
check("every input text and option matches the bank", True)

# ID lookup is the normal downstream access path. It must not silently reorder options.
def get_question(item_id):
    rows = index.loc[index["item_id"].eq(item_id)].sort_values("answer_index")
    if rows.empty:
        raise KeyError(item_id)
    return rows, vectors[rows["embedding_row"].to_numpy()]
for item_id, item in items.items():
    rows, values = get_question(item_id)
    assert rows["answer_index"].tolist() == list(range(len(item["options"])))
    assert values.shape == (len(item["options"]), 1024)
try:
    get_question("not_a_question")
except KeyError:
    check("unknown question rejected", True)
else:
    raise AssertionError("Unknown ID was accepted")
check("all 441 questions retrievable in option order", True)

# Exact-vector search diagnoses collisions/order corruption, not semantic accuracy.
correct = 0
for start in range(0, len(vectors), 128):
    found = (vectors[start:start + 128] @ vectors.T).argmax(axis=1)
    correct += int((found == np.arange(start, start + len(found))).sum())
check("all vectors retrieve their own indexed option", correct == len(vectors))
print(f"All {len(records):,} texts, vectors and option lookups passed.")


/Users/louismcauliffe/code/projects/AS_Tech_Task/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All 1,877 texts, vectors and option lookups passed.


## 1. Join permitted human responses
Read only item IDs, answer indices and missing flags from the three approved response tables. Missing answers must remain unmatched.


In [2]:
join_results = []
lookup = index[["item_id", "answer_index", "embedding_row"]]
for name in ["shared_training_long", "development_anchors_long", "final_anchors_long"]:
    path = ROOT / "data/processed" / (name + ".parquet")
    expected_hash = read_json("outputs/data_quality_leakage_audit.json")["input_sha256"][str(path.relative_to(ROOT))]
    check("response checksum: " + name, sha256(path) == expected_hash)
    responses = pd.read_parquet(path, columns=["item_id", "answer_index", "is_missing"])
    joined = responses.merge(lookup, on=["item_id", "answer_index"], how="left", validate="many_to_one", indicator=True)
    check("response lookup: " + name,
          joined.loc[~joined["is_missing"], "_merge"].eq("both").all() and
          joined.loc[joined["is_missing"], "_merge"].eq("left_only").all())
    join_results.append({"partition": name, "matched_answers": int((~joined["is_missing"]).sum()),
                         "missing_answers_left_unmatched": int(joined["is_missing"].sum())})
display(pd.DataFrame(join_results))


,partition,matched_answers,missing_answers_left_unmatched
0,shared_training_long,712827,113
1,development_anchors_long,27576,24
2,final_anchors_long,9198,2


## 2. Fresh model and independent pooling
Recompute a fixed training-only sample directly from token states, bypassing `SentenceTransformer.encode`. Check token lengths for every saved input and retrieve the fresh vectors against the saved matrix.


In [3]:
snapshot = ROOT / config["cache_directory"] / ("models--" + config["model_id"].replace("/", "--")) / "snapshots" / config["revision"]
for name, digest in audit["model_file_sha256"].items():
    check("model checksum: " + name, sha256(snapshot / name) == digest)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model = SentenceTransformer(str(snapshot), device=device, local_files_only=True, trust_remote_code=False,
                           model_kwargs={"dtype": torch.float32, "attn_implementation": "sdpa"})
model.eval()
model.requires_grad_(False)
model.tokenizer.padding_side = "left"
check("frozen last-token encoder", not model.training and not any(p.requires_grad for p in model.parameters()) and model[1].pooling_mode == "lasttoken")
texts = index["input_text"].tolist()
lengths = [len(ids) for ids in model.tokenizer(texts, truncation=False, padding=False)["input_ids"]]
check("all token counts match", lengths == index["token_count"].tolist())
check("no truncated inputs", max(lengths) <= config["max_sequence_length"])

def direct_encode(texts):
    chunks = []
    with torch.inference_mode():
        for offset in range(0, len(texts), 8):
            tokens = model.tokenizer(texts[offset:offset + 8], padding=True, truncation=False, return_tensors="pt").to(device)
            assert tokens["attention_mask"][:, -1].eq(1).all()
            hidden = model[0].auto_model(**tokens).last_hidden_state
            pooled = torch.nn.functional.normalize(hidden[:, -1, :], p=2, dim=1)
            chunks.append(pooled.cpu().numpy())
    return np.concatenate(chunks)

sample = []
for q in ["QID11", "QID29_1", "QID84_4", "QID250_1", "QID279_1", "QID119", "QID287_4", "QID235_1", "QID238_22", "QID31"]:
    rows, _ = get_question(q)
    sample.extend(rows["embedding_row"].iloc[[0, -1]].tolist())
rng = np.random.default_rng(20261023)
sample = sorted(set(sample + rng.choice(index.loc[index["question_role"].eq("training"), "embedding_row"], size=12, replace=False).tolist()))
assert index.iloc[sample]["question_role"].eq("training").all()
fresh = direct_encode([texts[j] for j in sample])
cosines = np.sum(fresh * vectors[sample], axis=1)
retrieved = (fresh @ vectors.T).argmax(axis=1)
check("fresh direct pooling reproduces saved vectors", np.all(cosines > 0.99999))
check("fresh vectors retrieve the exact original options", retrieved.tolist() == sample)
print(f"Fresh reference check: {len(sample)} candidates; minimum cosine {cosines.min():.8f}.")


Fresh reference check: 31 candidates; minimum cosine 1.00000000.


## 3. Small semantic diagnostic
Ten paraphrases, fixed before running, use training questions only. Keep the question context but omit candidate option numbers to test meaning beyond matching answer codes. Rank each paraphrase against that question's options. These examples are not a benchmark and do not tune the encoder.


In [4]:
probes = [
    ("QID235_1", 4, "I strongly agree that it matters to me that the products I use do not damage the environment."),
    ("QID235_1", 0, "I strongly disagree that it matters to me that the products I use do not damage the environment."),
    ("QID287_4", 4, "I am strongly in favour of this Medicare for All proposal."),
    ("QID287_4", 0, "I am strongly against this Medicare for All proposal."),
    ("QID250_1", 0, "I choose the gamble: an equal chance of receiving six dollars or nothing."),
    ("QID250_1", 1, "I choose to receive fifty cents for certain instead of the gamble."),
    ("QID279_1", 0, "I choose a fifty-fifty gamble between losing eight dollars and winning seven dollars."),
    ("QID279_1", 1, "I choose a guaranteed zero dollars, without taking the gamble."),
    ("QID119", 0, "I keep none of the money and return all twelve dollars to the sender."),
    ("QID119", 12, "I keep all twelve dollars for myself and return nothing to the sender."),
]
assert all(items[q]["role"] == "training" for q, _, _ in probes)
probe_texts = [f"Instruct: {config['instruction']}\nQuery: {items[q]['question_text']}\n\nCandidate answer:\n{paraphrase}"
               for q, _, paraphrase in probes]
probe_vectors = direct_encode(probe_texts)
semantic_results = []
for (q, expected_index, paraphrase), vector in zip(probes, probe_vectors):
    options, candidates = get_question(q)
    scores = candidates @ vector
    order = np.argsort(-scores, kind="stable")
    predicted = int(options.iloc[order[0]]["answer_index"])
    expected_position = options["answer_index"].tolist().index(expected_index)
    other_scores = np.delete(scores, expected_position)
    semantic_results.append({"item_id": q, "paraphrase": paraphrase, "expected_answer_index": expected_index,
                             "retrieved_answer_index": predicted, "correct": predicted == expected_index,
                             "expected_rank": int(np.flatnonzero(order == expected_position)[0]) + 1,
                             "expected_margin_over_best_alternative": float(scores[expected_position] - other_scores.max())})
display(pd.DataFrame(semantic_results)[["item_id", "expected_answer_index", "retrieved_answer_index", "expected_rank", "correct"]])
semantic_correct = sum(r["correct"] for r in semantic_results)
print(f"Paraphrase retrieval: {semantic_correct}/{len(probes)} intended options ranked first.")


,item_id,expected_answer_index,retrieved_answer_index,expected_rank,correct
0,QID235_1,4,4,1,True
1,QID235_1,0,0,1,True
2,QID287_4,4,4,1,True
3,QID287_4,0,0,1,True
4,QID250_1,0,0,1,True
5,QID250_1,1,1,1,True
6,QID279_1,0,0,1,True
7,QID279_1,1,1,1,True
8,QID119,0,12,2,False
9,QID119,12,12,1,True


Paraphrase retrieval: 9/10 intended options ranked first.


## 4. Save the findings


In [5]:
for name, entry in audit["files"].items():
    check("embedding file unchanged: " + name, sha256(ROOT / name) == entry["sha256"])
report = {
    "status": "passed" if semantic_correct == len(probes) else "structural_checks_passed_semantic_caveats",
    "checks_passed": len(checks), "checks": checks,
    "source_embedding_audit_sha256": sha256(ROOT / "outputs/question_answer_embedding_audit.json"),
    "all_candidate_self_retrieval": {"correct": correct, "total": len(vectors)},
    "permitted_response_joins": join_results,
    "independent_pooling": {"sample_rows": sample, "minimum_cosine": float(cosines.min()),
                            "exact_option_retrieval_correct": int((retrieved == sample).sum()), "total": len(sample)},
    "semantic_diagnostic": {"correct": semantic_correct, "total": len(probes), "results": semantic_results,
                            "limitation": "Small hand-written training-only diagnostic; not evidence of survey prediction accuracy. No encoder tuning or feature changes performed."},
    "device": device, "heldout_response_tables_loaded": False,
    "embedding_files_modified": False,
}
path = ROOT / "outputs/embedding_retrieval_checks.json"
path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n")
print(f"Saved {path.relative_to(ROOT)}; {len(checks)} structural/reproducibility checks passed.")


Saved outputs/embedding_retrieval_checks.json; 39 structural/reproducibility checks passed.
